# Week 6 Final AI Assignment: Student Performance Prediction

**Objective:** Build a complete machine-learning workflow to estimate students' final scores from study habits, attendance, previous scores, and selected background indicators.

> **Dataset note:** `student_performance.csv` is a reproducible, synthetic educational dataset created for this assignment. It is not real student data and should not be used to make consequential decisions about actual students. For a real deployment, use an ethically sourced, consented dataset and assess fairness.

## Workflow
Problem definition → data loading → cleaning → EDA → preprocessing → train/test split → Random Forest regression → predictions → evaluation → conclusion.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_PATH = Path("data/student_performance.csv")  # Run notebook from project root
df = pd.read_csv(DATA_PATH)
print("Dataset loaded:", df.shape)


## 2. Inspect and clean the dataset

In [ ]:
print("First five records:")
display(df.head())
print("Last five records:")
display(df.tail())
print("Shape (rows, columns):", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())

rows_before = len(df)
missing_before = int(df.isna().sum().sum())
df_clean = df.drop_duplicates().copy()

numeric_cols_all = df_clean.select_dtypes(include=np.number).columns
categorical_cols_all = df_clean.select_dtypes(exclude=np.number).columns
for col in numeric_cols_all:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in categorical_cols_all:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode().iloc[0])

print("Rows before cleaning:", rows_before)
print("Rows after removing duplicates:", len(df_clean))
print("Missing cells before:", missing_before)
print("Missing cells after:", int(df_clean.isna().sum().sum()))
display(df_clean.head())


## 3. Exploratory Data Analysis (EDA)
Each plot explores a different aspect of the dataset. Correlation shows association, not causation.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df_clean["final_score"], bins=20, edgecolor="black")
plt.title("Distribution of Final Scores")
plt.xlabel("Final score (0–100)")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

# Insight: this chart shows the score range and where scores are concentrated.


In [ ]:
study_bins = pd.cut(df_clean["hours_studied"], bins=[-0.1, 2, 4, 6, 8, 10, 12])
mean_by_hours = df_clean.groupby(study_bins, observed=False)["final_score"].mean()
mean_by_hours.plot(kind="bar", edgecolor="black", figsize=(7, 4))
plt.title("Average Final Score by Study-Hour Band")
plt.xlabel("Study hours band")
plt.ylabel("Mean final score")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

# Insight: compare average scores across study-hour groups; this is descriptive, not proof of causality.


In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df_clean["attendance_pct"], df_clean["final_score"], alpha=0.55)
plt.title("Attendance vs Final Score")
plt.xlabel("Attendance (%)")
plt.ylabel("Final score")
plt.tight_layout()
plt.show()

# Insight: look for a trend or spread in scores as attendance changes.


In [ ]:
numeric_corr = df_clean.select_dtypes(include=np.number).corr()
plt.figure(figsize=(7, 5))
plt.imshow(numeric_corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_corr.columns)), numeric_corr.columns, rotation=45, ha="right")
plt.yticks(range(len(numeric_corr.columns)), numeric_corr.columns)
plt.title("Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

# Insight: values nearer +1 indicate positive linear association; nearer -1 indicate negative association.


In [ ]:
df_clean.boxplot(column="final_score", by="tutoring", figsize=(7, 4))
plt.title("Final Score by Tutoring Participation")
plt.suptitle("")
plt.xlabel("Tutoring")
plt.ylabel("Final score")
plt.tight_layout()
plt.show()

# Insight: compare medians, spread, and outliers between tutoring groups.


## 4. Define features, target, and preprocessing

In [ ]:
TARGET = "final_score"
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 5. Train the model

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=250, min_samples_leaf=2, random_state=RANDOM_STATE
    ))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Model training complete.")


### Why Random Forest Regression?
Random Forest combines predictions from many decision trees, can model nonlinear relationships, and handles mixed numerical/categorical inputs after preprocessing. It is a useful baseline for tabular data. It is not guaranteed to outperform every alternative; cross-validation and baseline comparisons are recommended.

## 6. Evaluate the model

In [ ]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Value": [mae, mse, rmse, r2]
})
display(metrics.round(3))


In [ ]:
sample_n = min(45, len(y_test))
plt.figure(figsize=(8, 4))
plt.plot(np.arange(sample_n), y_test.iloc[:sample_n].to_numpy(), marker="o", label="Actual")
plt.plot(np.arange(sample_n), predictions[:sample_n], marker="x", label="Predicted")
plt.title("Actual vs Predicted Scores (Test Sample)")
plt.xlabel("Test sample index")
plt.ylabel("Final score")
plt.legend()
plt.tight_layout()
plt.show()


**Metric interpretation**
- **MAE:** average absolute error in score points; lower is better.
- **MSE:** average squared error; penalizes larger errors more heavily.
- **RMSE:** square root of MSE, in score points.
- **R²:** relative fit against predicting the test-set mean; 1 is ideal, 0 is similar to the mean baseline, and it can be negative.

Results depend on the random split and this synthetic dataset. Do not treat these metrics as evidence of real-world student outcomes.

## 7. Feature importance

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = model.named_steps["regressor"].feature_importances_
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(importance_df.head(10))
top = importance_df.head(10).sort_values("importance")
plt.figure(figsize=(7, 4))
plt.barh(top["feature"], top["importance"], edgecolor="black")
plt.title("Top 10 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 8. Generate sample predictions and save outputs

In [ ]:
results = X_test.copy()
results["actual_final_score"] = y_test.to_numpy()
results["predicted_final_score"] = np.round(predictions, 2)

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
results.to_csv(output_dir / "test_predictions.csv", index=False)
metrics.to_csv(output_dir / "evaluation_metrics.csv", index=False)
importance_df.to_csv(output_dir / "feature_importance.csv", index=False)
print("Saved predictions and evaluation outputs to:", output_dir.resolve())
display(results.head(10))


## 9. Conclusion and future scope
This notebook demonstrates the end-to-end supervised regression workflow. Future improvements include using a real, documented dataset; comparing linear regression, gradient boosting, and random forest with cross-validation; tuning hyperparameters; checking error by subgroup; and discussing privacy, consent, and fairness before any educational use.

**Important:** this educational example is not a validated tool for grading, admissions, intervention, or other consequential decisions.